# 🔥 MapBiomas Fire — Sentinel-2 Monitor
## Piloto Perú · Google Colab

**Flujo de campaña semanal:**  
Semana 1 → Mosaico + Muestras + Entrenamiento  
Semana 2 → Clasificación + Publicación

| Módulo | Descripción |
|--------|-----------|
| M0 | Autenticación y Configuración |
| M1 | Generador de Mosaicos Sentinel-2 |
| M2 | Gestor de Muestras |
| M3 | Entrenamiento del Modelo DNN |
| M4 | Clasificación por Campaña |
| M5 | Máscara LULC + Publicación |

---
> **Resultado:** Píxeles quemados = `dayOfYear` (día juliano del mes) · No quemados = `0`  
> **Máscara LULC:** clases 26, 22, 33, 24 (MapBiomas Perú Colección 2) — aplicada post-clasificación

## ⚙️ M0 — Instalación y Autenticación

In [ ]:
# ── Instalar dependencias (ejecutar una vez por sesión)
!pip install -q earthengine-api gcsfs rasterio scipy
!pip install -q tensorflow==2.13.0  # TF2 con compatibilidad TF1 vía compat.v1
!apt-get -qq install -y gdal-bin python3-gdal > /dev/null

In [ ]:
import sys, os

# ── Montar Google Drive (para persistir archivos entre sesiones)
from google.colab import drive
drive.mount('/content/drive')

# ── Clonar / apuntar al repositorio
# Opción A: clonar de GitHub
# !git clone https://github.com/SEU_ORG/fire_monitor.git /content/fire_monitor

# Opción B: usar Drive
REPO_PATH = '/content/drive/MyDrive/fire_monitor'  # ajustar según sea necesario
SRC_PATH  = os.path.join(REPO_PATH, 'src', 'colab')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'✅ sys.path configurado: {SRC_PATH}')

In [ ]:
# ── Autenticación GEE + GCS
from google.colab import auth as colab_auth
colab_auth.authenticate_user()  # autentica ADC para GCS automáticamente

from M0_auth_config import CONFIG, authenticate, print_config
authenticate()   # inicializa ee con project
print_config()

## 🛰️ M1 — Generador de Mosaicos Sentinel-2

In [ ]:
# ── Interfaz de monitoreo y envío de mosaicos
# Funciones:
#   🔍 Verificar Estado → comprueba qué mosaicos ya están en GCS/GEE
#   🚀 Enviar           → envía tareas de GEE (Export.image para GCS + Asset)
#   🗺️ Ensamblar        → construye VRT → COG del mosaico del país (después de que las tareas hayan concluido)
#
# Configuración en M0:
#   tile_size_deg = 0.83° (~92km × 92km = 1/4 de escena Landsat)
#   Mosaico mensual: enmascarado por el buffer de focos de INPE (Sudamérica)
#   Mosaico anual: país completo (sin máscara de focos)
#   Escala byte: dividir(100) → 0–100 | dayOfYear → int16 (sin conversión)

from M1_mosaic_generator import run_ui as mosaic_ui
mosaic_interface = mosaic_ui()

In [ ]:
# ── (Opcional) Monitorear tareas de GEE enviadas
import ee
tasks = ee.batch.Task.list()
for t in tasks[:10]:
    s = t.status()
    print(f"  {s['state']:12s} | {s['description']}")

## 🧬 M2 — Gestor de Muestras

In [ ]:
# ── Selección de muestras vectoriales + bandas del modelo
#
# Muestras exportadas por el Toolkit (FeatureCollection en GEE)
# Esquema: label, year, month, period, region, version, campaign, source
#
# Selectores disponibles:
#   Colección → FeatureCollection en GEE
#   Versión   → versión de la muestra (p. ej. v1)
#   Período   → mensual / anual / ambos
#   Regiones  → filtro por región
#   Años      → filtro por años
#
# Bandas (casillas de verificación):
#   ✅ Predeterminadas: red, nir, swir1, swir2
#   ⬜ Opcionales: blue, green, dayOfYear

from M2_sample_manager import run_ui as sample_ui
sample_interface = sample_ui()

In [ ]:
# ── Recuperar la selección confirmada de M2
sample_fc, selected_bands = sample_interface.get_selection()
print(f'✅ {sample_fc.size().getInfo():,} muestras | Bandas: {selected_bands}')

## 🧠 M3 — Entrenamiento del Modelo DNN

In [ ]:
# ── Entrenamiento de la DNN con las bandas seleccionadas en M2
#
# NUM_INPUT = dinámico (4–7 bandas según la selección)
# Arquitectura predeterminada: [7, 14, 7, 14, 7] (totalmente conectada, ReLU + dropout)
# Resultado guardado en GCS: models/{version}/{region}/
#   - weights.npz
#   - hyperparameters.json (incluye bandas, muestras, fecha, normalización)

from M3_model_trainer import run_ui as trainer_ui
trainer_interface = trainer_ui(
    sample_fc=sample_fc,
    selected_bands=selected_bands
)

## 🔥 M4 — Clasificación por Campaña

In [ ]:
# ── Clasificación por cuadrícula dinámica y por región
#
# Pipeline por fragmento:
#   1. Descargar fragmento COG de GCS
#   2. Inferencia DNN con las bandas del modelo
#   3. Filtros morfológicos (apertura + cierre + sieve)
#   4. Píxeles quemados → reciben el valor dayOfYear (no el valor 1)
#   5. Subida a GCS chunks/classifications/
#
# Cuadrícula: ~0.83° × 0.83° (~92km = 1/4 de escena Landsat)
# Enmascaramiento de periferia por región aplicado automáticamente
# Máscara LULC (26/22/33/24) → aplicada en M5, NO aquí

from M4_classifier import run_ui as classifier_ui
classifier_interface = classifier_ui()

In [ ]:
# ── (Opcional) Ejecutar la campaña programáticamente (sin interfaz de usuario)
# from M4_classifier import run_classification_campaign
# results = run_classification_campaign(
#     year    = 2024,
#     months  = [7, 8, 9],
#     regions = ['peru_r1_amazonia'],
#     version = 'v1'
# )

## 📢 M5 — Máscara LULC y Publicación

In [ ]:
# ── Post-clasificación → publicación en GEE
#
# Pipeline:
#   1. 🔍 Verificar Cobertura → comprueba los fragmentos clasificados
#   2. 🗺️ Ensamblar Mosaico → VRT → COG del país (GCS mosaics/)
#   3. 📢 Aplicar Máscara + Publicar →
#        - Máscara LULC: clases 26, 22, 33, 24 (MapBiomas Perú Colección 2)
#        - Elimina píxeles aislados (< 4 píxeles conectados)
#        - Exportar a GEE Asset con versión + metadatos completos
#
# Versión:
#   🟡 Borrador → nombre: burned_area_..._draft (revisión interna)
#   🟢 Final    → nombre: burned_area_...       (conjunto de datos público)

from M5_publisher import run_ui as publisher_ui
publisher_interface = publisher_ui()

---
## 📋 Flujo Operativo de la Campaña

```
SEMANA 1 (primeros ~5 días hábiles)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Día 1 │ M0: autenticar + configurar
Día 1 │ M1: verificar estado → enviar mosaico mensual (tareas de GEE)
Día 2 │ M1: verificar tareas concluidas → construir mosaico del país (VRT→COG)
Día 2 │ M2: seleccionar muestras + confirmar bandas del modelo
Día 3 │ M3: entrenar modelo (saltar si el modelo existente está bien)
Día 3 │ M4: enviar clasificación por región + cuadrícula

SEMANA 2 (hasta el día 10)
━━━━━━━━━━━━━━━━━━━━
Día 6 │ M4: verificar fragmentos clasificados
Día 6 │ M5: construir mosaico de clasificación del país
Día 7 │ M5: revisión → borrador o final → publicar en GEE
```
